# EZhire - Multi-Model Resume-Job Semantic Similarity Scoring
> Three-model training pipeline: all-mpnet-base-v2 | all-roberta-large-v1 | jina-embeddings-v2

**Pipeline:**
1. Install & Import
2. Load dataset
3. Preprocess
4. Shared chunking infrastructure (1-sentence doc context, 150-char overlap)
5. Train & evaluate **Model A** — all-mpnet-base-v2
6. Train & evaluate **Model B** — all-roberta-large-v1
7. Select best SBERT model (A vs B by Pearson)
8. Train & evaluate **Model C** — jina-embeddings-v2 (no chunking, full text)
9. Final ensemble: best SBERT + TF-IDF (CV grid-search weight)
10. Full metrics comparison
11. Gradio dashboard

**Runtime:** GPU > T4

In [1]:
!pip install -q sentence-transformers datasets gradio scikit-learn plotly nltk PyMuPDF einops

In [2]:
import os, re, warnings
import numpy as np
import pandas as pd
import gradio as gr
import plotly.graph_objects as go
import nltk
import fitz
import torch
from datasets import load_dataset
from sentence_transformers import SentenceTransformer, util, InputExample, losses
from sentence_transformers.evaluation import SentenceEvaluator
from torch.utils.data import DataLoader
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import mean_absolute_error, ndcg_score, r2_score
from sklearn.model_selection import StratifiedKFold
from scipy.stats import spearmanr, pearsonr
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from collections import Counter

warnings.filterwarnings('ignore')
nltk.download('stopwords', quiet=True)
nltk.download('punkt',     quiet=True)
nltk.download('punkt_tab', quiet=True)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
STOP_WORDS = set(stopwords.words('english'))
print(f'Device : {DEVICE}')
print('Imports OK')

Device : cuda
Imports OK


/tmp/ipykernel_17143/657359280.py:10: DeprecationWarning: Importing from 'sentence_transformers.losses' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.losses' instead.
  from sentence_transformers import SentenceTransformer, util, InputExample, losses
/tmp/ipykernel_17143/657359280.py:11: DeprecationWarning: Importing from 'sentence_transformers.evaluation' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.evaluation' instead.
  from sentence_transformers.evaluation import SentenceEvaluator


In [3]:
ds       = load_dataset('0xnbk/resume-ats-score-v1-en')
df_train = ds['train'].to_pandas()
df_val   = ds['validation'].to_pandas()
print(f'Train: {len(df_train)} | Val: {len(df_val)}')
print(f'Columns: {list(df_train.columns)}')
print(f'ATS range: {df_train["ats_score"].min():.1f} - {df_train["ats_score"].max():.1f}')
print(df_train['original_label'].value_counts().to_string())

Train: 5099 | Val: 1275
Columns: ['text', 'ats_score', 'original_label']
ATS range: 19.2 - 90.0
original_label
No Fit           2565
Potential Fit    1273
Good Fit         1261


In [4]:
ATS_MIN, ATS_MAX = 18.3, 90.7

def split_sep(text):
    if not isinstance(text, str): text = str(text)
    if '[SEP]' in text:
        a, b = text.split('[SEP]', 1)
        return a.strip(), b.strip()
    mid = len(text)//2
    return text[:mid].strip(), text[mid:].strip()

def raw_text(text):
    if not isinstance(text, str): text = str(text)
    text = text.replace('\r\n','\n').replace('\r','\n')
    lines = [re.sub(r'[ \t\f\v]+',' ',l).strip() for l in text.split('\n')]
    return ' '.join(l for l in lines if l)

def clean_text(text):
    if not isinstance(text, str): text = str(text)
    text = re.sub(r'[^a-z0-9\s]',' ', text.lower())
    tokens = word_tokenize(re.sub(r'\s+',' ', text).strip())
    return ' '.join(t for t in tokens if t not in STOP_WORDS and len(t)>1)

def normalize_score(series):
    v = pd.to_numeric(series, errors='coerce').astype(float)
    return ((v - ATS_MIN) / (ATS_MAX - ATS_MIN)).clip(0,1)

def denormalize_score(v):
    return np.asarray(v, float) * (ATS_MAX - ATS_MIN) + ATS_MIN

def build_df(src):
    splits = src['text'].apply(split_sep)
    ats    = pd.to_numeric(src['ats_score'], errors='coerce').fillna(ATS_MIN)
    return pd.DataFrame({
        'resume_raw'    : splits.apply(lambda x: raw_text(x[0])),
        'jd_raw'        : splits.apply(lambda x: raw_text(x[1])),
        'resume_clean'  : splits.apply(lambda x: clean_text(x[0])),
        'jd_clean'      : splits.apply(lambda x: clean_text(x[1])),
        'original_label': src['original_label'].values,
        'ats_score_raw' : ats.values,
        'ground_truth'  : normalize_score(ats).values
    }).dropna(subset=['resume_raw','jd_raw']).reset_index(drop=True)

df_tr = build_df(df_train)
df_vl = build_df(df_val)
print(f'Train: {len(df_tr)} | Val: {len(df_vl)}')
print(f'GT range train: {df_tr["ground_truth"].min():.3f} - {df_tr["ground_truth"].max():.3f}')
df_tr.head(2)

Train: 5099 | Val: 1275
GT range train: 0.012 - 0.991


,resume_raw,jd_raw,resume_clean,jd_clean,original_label,ats_score_raw,ground_truth
0,SummaryI am seeking a position wherein I may u...,"iness processes to ensure functionality, compl...",summaryi seeking position wherein may use prov...,iness processes ensure functionality completen...,Good Fit,80.6,0.860497
1,ProfileHighly motivated Sales Associate with e...,"bases, Database, e-Commerce, e-Business, Engli...",profilehighly motivated sales associate extens...,bases database commerce business english featu...,No Fit,24.3,0.082873


## Shared Chunking Infrastructure
Used by Model A and Model B (512-token models). Jina skips this entirely.

**Key settings:**
- Doc context prefix: 1 sentence max (first sentence of the document)
- Chunk overlap: 150 chars (approx 38 tokens)
- Section-aware splitting before token chunking

In [ ]:
# ── Chunking Hyperparameters ──────────────────────────────────────────
# Budget: 512 (model limit) - ~50 (prefix) - 2 (special tokens) = 460 body tokens
# make_text_chunks passes max_seq_length=512 so token_chunk_body
# subtracts prefix_tokens internally — no double subtraction.

CHUNK_OVERLAP          = 38    # ~150 chars / 4 chars-per-token
MAX_RESUME_CHUNKS      = 10
MAX_JD_CHUNKS          = 8
MAX_TRAIN_PAIRS        = 4
MAX_EVAL_PAIRS         = 2
TOP_K_CHUNK_PAIRS      = 5
ENCODE_BATCH           = 16 if DEVICE == 'cuda' else 8
DOC_EVAL_N             = 200
RANDOM_SEED            = 42

np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(RANDOM_SEED)

from sentence_transformers.evaluation import SentenceEvaluator  # fix Issue 4

# ── 1-Sentence Doc Context Extractor ─────────────────────────────────
def extract_first_sentence(text, max_chars=150):
    t = raw_text(text)
    m = re.search(r'(?<=[a-zA-Z0-9])[.!?]', t)
    if m and m.start() > 10:
        sentence = t[:m.start()+1].strip()
    else:
        sentence = t[:max_chars].strip()
    return sentence[:max_chars]

# ── Token Chunker ─────────────────────────────────────────────────────
def token_chunk_body(body, tokenizer,
                     max_tokens=512,
                     overlap=CHUNK_OVERLAP,
                     prefix_tokens=0):
    body = raw_text(body)
    if not body:
        return []
    ids = tokenizer.encode(body, add_special_tokens=False, truncation=False)

    # Body budget = model limit - prefix tokens - 2 special tokens [CLS][SEP]
    effective_max = max(64, max_tokens - prefix_tokens - 2)

    if len(ids) <= effective_max:
        return [body]

    overlap = min(overlap, effective_max // 2)
    step    = effective_max - overlap
    chunks  = []
    for start in range(0, len(ids), step):
        piece = ids[start:start + effective_max]
        chunk = raw_text(tokenizer.decode(piece, skip_special_tokens=True))
        if chunk:
            chunks.append(chunk)
        if start + effective_max >= len(ids):
            break
    return chunks

# ── Main Chunk Builder ────────────────────────────────────────────────
def make_text_chunks(text, tokenizer,
                     model_max_tokens=512,   # true model hard limit
                     overlap=CHUNK_OVERLAP,
                     max_chunks=MAX_RESUME_CHUNKS,
                     source_label='DOCUMENT'):
    text = raw_text(text)
    if not text:
        return []

    doc_ctx = extract_first_sentence(text)
    prefix  = f'[DOC]: {doc_ctx} [{source_label}] ' if doc_ctx else f'[{source_label}] '

    # Measure prefix cost once — subtracted inside token_chunk_body
    prefix_tokens = len(tokenizer.encode(prefix, add_special_tokens=False))

    body_chunks = token_chunk_body(
        text, tokenizer,
        max_tokens    = model_max_tokens,  # pass full limit, not pre-subtracted
        overlap       = overlap,
        prefix_tokens = prefix_tokens      # subtracted once, inside token_chunk_body
    )

    chunks = [f'{prefix}{c}' for c in body_chunks]

    if len(chunks) > max_chunks:
        keep   = np.linspace(0, len(chunks)-1, max_chunks, dtype=int).tolist()
        chunks = [chunks[i] for i in keep]

    fallback = text[:2000]
    return chunks or [f'{prefix}{fallback}']

# ── Top Chunk-Pair Selector ───────────────────────────────────────────
def select_top_chunk_pairs(r_chunks, j_chunks, max_pairs=MAX_TRAIN_PAIRS):
    if len(r_chunks) * len(j_chunks) <= max_pairs:
        return [(r, j) for r in r_chunks for j in j_chunks]
    try:
        vec  = TfidfVectorizer(stop_words='english', ngram_range=(1,2), max_features=5000)
        mat  = vec.fit_transform(r_chunks + j_chunks)
        sims = cosine_similarity(mat[:len(r_chunks)], mat[len(r_chunks):])
        ranked = np.argsort(sims.reshape(-1))[::-1]
    except Exception:
        ranked = range(len(r_chunks) * len(j_chunks))
    selected, seen = [], set()
    for idx in ranked:
        i, j = int(idx) // len(j_chunks), int(idx) % len(j_chunks)
        if (i, j) not in seen:
            selected.append((r_chunks[i], j_chunks[j]))
            seen.add((i, j))
        if len(selected) >= max_pairs:
            break
    return selected or [(r_chunks[0], j_chunks[0])]

# ── Build InputExamples ───────────────────────────────────────────────
def build_chunked_examples(frame, tokenizer,
                           max_pairs=MAX_TRAIN_PAIRS,
                           model_max_tokens=512):
    examples   = []
    total_rows = len(frame)
    for idx, (_, row) in enumerate(frame.iterrows()):
        rc = make_text_chunks(row['resume_raw'], tokenizer,
                              model_max_tokens=model_max_tokens,
                              max_chunks=MAX_RESUME_CHUNKS,
                              source_label='RESUME')
        jc = make_text_chunks(row['jd_raw'], tokenizer,
                              model_max_tokens=model_max_tokens,
                              max_chunks=MAX_JD_CHUNKS,
                              source_label='JOB')
        for r, j in select_top_chunk_pairs(rc, jc, max_pairs):
            examples.append(InputExample(texts=[r, j],
                                         label=float(row['ground_truth'])))
        if (idx+1) % 500 == 0:
            print(f'  Built examples for {idx+1}/{total_rows} rows '
                  f'({len(examples)} chunk-pairs so far)')
    return examples

# ── Aggregate chunk similarities -> doc score ─────────────────────────
def aggregate_chunk_sims(sims, top_k=TOP_K_CHUNK_PAIRS):
    sims = np.asarray(sims, float)
    if sims.size == 0: return 0.0
    flat     = sims.reshape(-1)
    k        = min(top_k, len(flat))
    top_mean = float(np.partition(flat, -k)[-k:].mean())
    coverage = float((sims.max(axis=0).mean() + sims.max(axis=1).mean()) / 2)
    return float(np.clip(0.75 * top_mean + 0.25 * coverage, 0.0, 1.0))

# ── Score a document pair ─────────────────────────────────────────────
def score_doc_pair(model, t1, t2, left='RESUME', right='JOB'):
    tok  = model.tokenizer
    mlen = model.max_seq_length
    c1   = make_text_chunks(t1, tok, model_max_tokens=mlen,
                            max_chunks=MAX_RESUME_CHUNKS, source_label=left)
    c2   = make_text_chunks(t2, tok, model_max_tokens=mlen,
                            max_chunks=MAX_JD_CHUNKS,    source_label=right)
    e1   = model.encode(c1, convert_to_tensor=True, normalize_embeddings=True,
                        batch_size=ENCODE_BATCH, show_progress_bar=False)
    e2   = model.encode(c2, convert_to_tensor=True, normalize_embeddings=True,
                        batch_size=ENCODE_BATCH, show_progress_bar=False)
    sims = util.cos_sim(e1, e2).detach().cpu().numpy()
    return aggregate_chunk_sims(sims)

# ── DocEvaluator ──────────────────────────────────────────────────────
class DocEvaluator(SentenceEvaluator):
    def __init__(self, frame, save_path, name='val'):
        self.frame     = frame.reset_index(drop=True)
        self.save_path = save_path
        self.name      = name
        self.best      = -np.inf

    def __call__(self, model, output_path=None, epoch=-1, steps=-1):
        preds = [score_doc_pair(model, r['resume_raw'], r['jd_raw'])
                 for _, r in self.frame.iterrows()]
        yt  = self.frame['ground_truth'].to_numpy(float)
        yp  = np.asarray(preds, float)
        pe  = float(pearsonr(yp, yt)[0]) if (len(yt)>1 and np.std(yt)>0 and np.std(yp)>0) else 0.0
        sp  = float(spearmanr(yp, yt).correlation) if len(yt)>1 else 0.0
        pe  = 0.0 if np.isnan(pe) else pe
        sp  = 0.0 if np.isnan(sp) else sp
        mae = mean_absolute_error(yt, yp)
        print(f'[{self.name}] Pearson={pe:+.4f} Spearman={sp:+.4f} MAE={mae:.4f}')
        if pe > self.best:
            self.best = pe
            model.save(self.save_path)
            print(f'  Saved best checkpoint -> {self.save_path}')
        return pe

# ── Sanity check ──────────────────────────────────────────────────────
def verify_chunk_lengths(examples, tokenizer, model_max):
    over = 0
    for ex in examples[:500]:
        for text in ex.texts:
            length = len(tokenizer.encode(text, add_special_tokens=True))
            if length > model_max:
                over += 1
    print(f'Chunks exceeding model max ({model_max}): {over}/1000')
    print('OK' if over == 0 else 'WARNING: reduce CHUNK_TOKENS or increase model_max')

print('Chunking infrastructure ready')
print('Budget: 512 (model) - prefix_tokens - 2 (special) = body tokens per chunk')
print('Doc context: 1 sentence max, capped at 150 chars')

## Model A — sentence-transformers/all-mpnet-base-v2
- 768-dim embeddings | 512-token limit | contextual chunking applied
- ~15-20 min on T4

In [ ]:
MPNET_NAME   = 'sentence-transformers/all-mpnet-base-v2'
MPNET_PATH   = './ezhire-mpnet'
EPOCHS_A     = 4
BATCH_A      = 16 if DEVICE == 'cuda' else 8
LR_A         = 2e-5
WARMUP_A     = 100
WD_A         = 0.01

print(f'Loading {MPNET_NAME}...')
mpnet_model = SentenceTransformer(MPNET_NAME, device=DEVICE)
mpnet_model.max_seq_length = 384

print('Building chunked training examples for mpnet...')
mpnet_train_ex = build_chunked_examples(df_tr, mpnet_model.tokenizer, MAX_TRAIN_PAIRS)
print(f'Training examples: {len(mpnet_train_ex)}')

mpnet_loader = DataLoader(mpnet_train_ex, shuffle=True, batch_size=BATCH_A)
mpnet_loss   = losses.CosineSimilarityLoss(mpnet_model)

val_sample_a  = df_vl.sample(n=min(DOC_EVAL_N, len(df_vl)), random_state=RANDOM_SEED)
mpnet_eval    = DocEvaluator(val_sample_a, save_path=MPNET_PATH, name='mpnet_val')

total_steps_a = len(mpnet_loader) * EPOCHS_A
warmup_a      = min(WARMUP_A, max(1, total_steps_a//10))
eval_steps_a  = max(100, len(mpnet_loader)//2)

print(f'Training: {EPOCHS_A} epochs | {len(mpnet_train_ex)} pairs | '
      f'batch {BATCH_A} | warmup {warmup_a}')

mpnet_model.fit(
    train_objectives  = [(mpnet_loader, mpnet_loss)],
    evaluator         = mpnet_eval,
    epochs            = EPOCHS_A,
    warmup_steps      = warmup_a,
    optimizer_params  = {'lr': LR_A},
    weight_decay      = WD_A,
    evaluation_steps  = eval_steps_a,
    output_path       = None,
    save_best_model   = False,
    show_progress_bar = True,
    use_amp           = (DEVICE == 'cuda')
)

if not os.path.exists(os.path.join(MPNET_PATH,'modules.json')):
    mpnet_model.save(MPNET_PATH)
mpnet_model = SentenceTransformer(MPNET_PATH, device=DEVICE)
mpnet_model.max_seq_length = 384
print('Model A (mpnet) fine-tuning complete')

Loading sentence-transformers/all-mpnet-base-v2...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Token indices sequence length is longer than the specified maximum sequence length for this model (491 > 384). Running this sequence through the model will result in indexing errors


Building chunked training examples for mpnet...
Training examples: 20298
Training: 4 epochs | 20298 pairs | batch 16 | warmup 100


Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

...

  Saved best checkpoint -> ./ezhire-mpnet
[mpnet_val] Pearson=+0.5520 Spearman=+0.5433 MAE=0.2455


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  Saved best checkpoint -> ./ezhire-mpnet
[mpnet_val] Pearson=+0.5539 Spearman=+0.5439 MAE=0.2446


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  Saved best checkpoint -> ./ezhire-mpnet
[mpnet_val] Pearson=+0.5172 Spearman=+0.5046 MAE=0.2414
[mpnet_val] Pearson=+0.5110 Spearman=+0.5050 MAE=0.2426
[mpnet_val] Pearson=+0.5132 Spearman=+0.5079 MAE=0.2428


In [ ]:
print('Evaluating Model A (mpnet) on full validation set...')
mpnet_preds = []
for i, row in df_vl.iterrows():
    mpnet_preds.append(score_doc_pair(mpnet_model, row['resume_raw'], row['jd_raw']))
    if (i+1) % 50 == 0: print(f'  {i+1}/{len(df_vl)}')

df_vl['mpnet_score'] = [round(s*100,2) for s in mpnet_preds]
yt = df_vl['ground_truth'].values
yp = df_vl['mpnet_score'].values
mpnet_pearson  = float(pearsonr(yp/100, yt)[0])
mpnet_spearman = float(spearmanr(yp/100, yt).correlation)
mpnet_mae      = float(mean_absolute_error(yt, yp/100))
mpnet_rmse     = float(np.sqrt(np.mean((denormalize_score(yt)-denormalize_score(yp/100))**2)))
mpnet_r2       = float(r2_score(yt, yp/100))

print(f'\nModel A (mpnet) Results:')
print(f'  Pearson  : {mpnet_pearson:+.4f}')
print(f'  Spearman : {mpnet_spearman:+.4f}')
print(f'  MAE_norm : {mpnet_mae:.4f}')
print(f'  RMSE_raw : {mpnet_rmse:.2f}')
print(f'  R2_raw   : {mpnet_r2:+.4f}')

## Model B — sentence-transformers/all-roberta-large-v1
- 1024-dim embeddings | 512-token limit | same contextual chunking applied
- ~25-30 min on T4 (larger model)

In [ ]:
ROBERTA_NAME  = 'sentence-transformers/all-roberta-large-v1'
ROBERTA_PATH  = './ezhire-roberta'
EPOCHS_B      = 4
BATCH_B       = 8 if DEVICE == 'cuda' else 4   # larger model needs smaller batch
LR_B          = 2e-5
WARMUP_B      = 100
WD_B          = 0.01

print(f'Loading {ROBERTA_NAME}...')
roberta_model = SentenceTransformer(ROBERTA_NAME, device=DEVICE)
roberta_model.max_seq_length = 384

print('Building chunked training examples for roberta...')
roberta_train_ex = build_chunked_examples(df_tr, roberta_model.tokenizer, MAX_TRAIN_PAIRS)
print(f'Training examples: {len(roberta_train_ex)}')

roberta_loader = DataLoader(roberta_train_ex, shuffle=True, batch_size=BATCH_B)
roberta_loss   = losses.CosineSimilarityLoss(roberta_model)

val_sample_b   = df_vl.sample(n=min(DOC_EVAL_N, len(df_vl)), random_state=RANDOM_SEED)
roberta_eval   = DocEvaluator(val_sample_b, save_path=ROBERTA_PATH, name='roberta_val')

total_steps_b  = len(roberta_loader) * EPOCHS_B
warmup_b       = min(WARMUP_B, max(1, total_steps_b//10))
eval_steps_b   = max(100, len(roberta_loader)//2)

print(f'Training: {EPOCHS_B} epochs | {len(roberta_train_ex)} pairs | '
      f'batch {BATCH_B} | warmup {warmup_b}')

roberta_model.fit(
    train_objectives  = [(roberta_loader, roberta_loss)],
    evaluator         = roberta_eval,
    epochs            = EPOCHS_B,
    warmup_steps      = warmup_b,
    optimizer_params  = {'lr': LR_B},
    weight_decay      = WD_B,
    evaluation_steps  = eval_steps_b,
    output_path       = None,
    save_best_model   = False,
    show_progress_bar = True,
    use_amp           = (DEVICE == 'cuda')
)

if not os.path.exists(os.path.join(ROBERTA_PATH,'modules.json')):
    roberta_model.save(ROBERTA_PATH)
roberta_model = SentenceTransformer(ROBERTA_PATH, device=DEVICE)
roberta_model.max_seq_length = 384
print('Model B (roberta) fine-tuning complete')

In [ ]:
print('Evaluating Model B (roberta) on full validation set...')
roberta_preds = []
for i, row in df_vl.iterrows():
    roberta_preds.append(score_doc_pair(roberta_model, row['resume_raw'], row['jd_raw']))
    if (i+1) % 50 == 0: print(f'  {i+1}/{len(df_vl)}')

df_vl['roberta_score'] = [round(s*100,2) for s in roberta_preds]
yt = df_vl['ground_truth'].values
yp = df_vl['roberta_score'].values
roberta_pearson  = float(pearsonr(yp/100, yt)[0])
roberta_spearman = float(spearmanr(yp/100, yt).correlation)
roberta_mae      = float(mean_absolute_error(yt, yp/100))
roberta_rmse     = float(np.sqrt(np.mean((denormalize_score(yt)-denormalize_score(yp/100))**2)))
roberta_r2       = float(r2_score(yt, yp/100))

print(f'\nModel B (roberta) Results:')
print(f'  Pearson  : {roberta_pearson:+.4f}')
print(f'  Spearman : {roberta_spearman:+.4f}')
print(f'  MAE_norm : {roberta_mae:.4f}')
print(f'  RMSE_raw : {roberta_rmse:.2f}')
print(f'  R2_raw   : {roberta_r2:+.4f}')

In [ ]:
# Automatically pick the SBERT model with higher Pearson on the validation set.
# This model becomes the SBERT component in the final hybrid ensemble.

print('Comparing Model A vs Model B...')
print(f'  mpnet   Pearson: {mpnet_pearson:+.4f}')
print(f'  roberta Pearson: {roberta_pearson:+.4f}')

if mpnet_pearson >= roberta_pearson:
    best_sbert_model = mpnet_model
    best_sbert_name  = 'mpnet'
    best_sbert_score_col = 'mpnet_score'
    print(f'\nBest SBERT -> Model A (mpnet) | Pearson={mpnet_pearson:+.4f}')
else:
    best_sbert_model = roberta_model
    best_sbert_name  = 'roberta'
    best_sbert_score_col = 'roberta_score'
    print(f'\nBest SBERT -> Model B (roberta) | Pearson={roberta_pearson:+.4f}')

print(f'This model will be used as the SBERT component in the final ensemble.')

## Model C — jinaai/jina-embeddings-v2-base-en
- 768-dim | **8,192-token native limit** — NO chunking needed
- Reads the full resume and JD in a single forward pass
- Trained separately with full-text InputExamples
- ~30 min on T4

In [ ]:
JINA_NAME    = 'jinaai/jina-embeddings-v2-base-en'
JINA_PATH    = './ezhire-jina'
EPOCHS_C     = 4
BATCH_C      = 8 if DEVICE == 'cuda' else 4
LR_C         = 2e-5
WARMUP_C     = 100
WD_C         = 0.01

print(f'Loading {JINA_NAME}...')
# trust_remote_code=True is required for Jina v2
jina_model = SentenceTransformer(JINA_NAME, trust_remote_code=True, device=DEVICE)
jina_model.max_seq_length = 8192
print(f'Jina max_seq_length: {jina_model.max_seq_length}')

# Full-text examples — NO chunking because Jina handles 8192 tokens natively
print('Building full-text training examples for Jina (no chunking)...')
jina_train_ex = [
    InputExample(texts=[row['resume_raw'], row['jd_raw']],
                 label=float(row['ground_truth']))
    for _, row in df_tr.iterrows()
]
print(f'Training examples: {len(jina_train_ex)} (one per row, full text)')

jina_loader  = DataLoader(jina_train_ex, shuffle=True, batch_size=BATCH_C)
jina_loss    = losses.CosineSimilarityLoss(jina_model)

# Document-level evaluator for Jina (full text, no chunking)
class JinaDocEvaluator(SentenceEvaluator):
    def __init__(self, frame, save_path, name='jina_val'):
        self.frame     = frame.reset_index(drop=True)
        self.save_path = save_path
        self.name      = name
        self.best      = -np.inf
    def __call__(self, model, output_path=None, epoch=-1, steps=-1):
        resumes = self.frame['resume_raw'].tolist()
        jds     = self.frame['jd_raw'].tolist()
        emb_r   = model.encode(resumes, batch_size=4, show_progress_bar=False,
                               convert_to_tensor=True, normalize_embeddings=True)
        emb_j   = model.encode(jds,     batch_size=4, show_progress_bar=False,
                               convert_to_tensor=True, normalize_embeddings=True)
        sims = util.cos_sim(emb_r, emb_j).diagonal().cpu().numpy()
        yt   = self.frame['ground_truth'].to_numpy(float)
        pe   = float(pearsonr(sims, yt)[0]) if np.std(sims)>0 and np.std(yt)>0 else 0.0
        sp   = float(spearmanr(sims, yt).correlation)
        pe   = 0.0 if np.isnan(pe) else pe
        sp   = 0.0 if np.isnan(sp) else sp
        mae  = float(mean_absolute_error(yt, sims))
        print(f'[{self.name}] Pearson={pe:+.4f} Spearman={sp:+.4f} MAE={mae:.4f}')
        if pe > self.best:
            self.best = pe
            model.save(self.save_path)
            print(f'  Saved best Jina checkpoint -> {self.save_path}')
        return pe

val_sample_c = df_vl.sample(n=min(DOC_EVAL_N, len(df_vl)), random_state=RANDOM_SEED)
jina_eval    = JinaDocEvaluator(val_sample_c, save_path=JINA_PATH)

total_steps_c = len(jina_loader) * EPOCHS_C
warmup_c      = min(WARMUP_C, max(1, total_steps_c//10))
eval_steps_c  = max(100, len(jina_loader)//2)

print(f'Training: {EPOCHS_C} epochs | {len(jina_train_ex)} full-text pairs | '
      f'batch {BATCH_C} | warmup {warmup_c}')

jina_model.fit(
    train_objectives  = [(jina_loader, jina_loss)],
    evaluator         = jina_eval,
    epochs            = EPOCHS_C,
    warmup_steps      = warmup_c,
    optimizer_params  = {'lr': LR_C},
    weight_decay      = WD_C,
    evaluation_steps  = eval_steps_c,
    output_path       = None,
    save_best_model   = False,
    show_progress_bar = True,
    use_amp           = (DEVICE == 'cuda')
)

if not os.path.exists(os.path.join(JINA_PATH,'modules.json')):
    jina_model.save(JINA_PATH)
jina_model = SentenceTransformer(JINA_PATH, trust_remote_code=True, device=DEVICE)
jina_model.max_seq_length = 8192
print('Model C (Jina) fine-tuning complete')

In [ ]:
print('Evaluating Model C (Jina) on full validation set (full text, no chunking)...')
jina_preds = []
for i, row in df_vl.iterrows():
    e1 = jina_model.encode(row['resume_raw'], convert_to_tensor=True,
                           normalize_embeddings=True, show_progress_bar=False)
    e2 = jina_model.encode(row['jd_raw'],     convert_to_tensor=True,
                           normalize_embeddings=True, show_progress_bar=False)
    jina_preds.append(float(util.cos_sim(e1.unsqueeze(0), e2.unsqueeze(0))[0][0]))
    if (i+1) % 50 == 0: print(f'  {i+1}/{len(df_vl)}')

df_vl['jina_score'] = [round(s*100,2) for s in jina_preds]
yt = df_vl['ground_truth'].values
yp = df_vl['jina_score'].values
jina_pearson  = float(pearsonr(yp/100,yt)[0])
jina_spearman = float(spearmanr(yp/100,yt).correlation)
jina_mae      = float(mean_absolute_error(yt,yp/100))
jina_rmse     = float(np.sqrt(np.mean((denormalize_score(yt)-denormalize_score(yp/100))**2)))
jina_r2       = float(r2_score(yt,yp/100))

print(f'\nModel C (Jina) Results:')
print(f'  Pearson  : {jina_pearson:+.4f}')
print(f'  Spearman : {jina_spearman:+.4f}')
print(f'  MAE_norm : {jina_mae:.4f}')
print(f'  RMSE_raw : {jina_rmse:.2f}')
print(f'  R2_raw   : {jina_r2:+.4f}')

## Final Hybrid Ensemble: Best SBERT + TF-IDF
Uses 5-fold Stratified CV grid search to find the optimal SBERT weight.

In [ ]:
# ── TF-IDF scoring (same as before) ─────────────────────────────────
def tfidf_score(t1, t2):
    try:
        m = TfidfVectorizer().fit_transform([t1,t2])
        return float(cosine_similarity(m[0:1],m[1:2])[0][0])
    except Exception: return 0.0

# ── Score TF-IDF for full val set ────────────────────────────────────
print('Computing TF-IDF scores...')
tfidf_preds = []
for i, row in df_vl.iterrows():
    tfidf_preds.append(round(tfidf_score(row['resume_clean'],row['jd_clean'])*100,2))
    if (i+1) % 100 == 0: print(f'  {i+1}/{len(df_vl)}')
df_vl['tfidf_score'] = tfidf_preds

# ── 5-fold CV grid search for optimal SBERT weight ───────────────────
print(f'\nGrid searching ensemble weight for best SBERT ({best_sbert_name})...')
best_sbert_preds = df_vl[best_sbert_score_col].values
tfidf_arr        = df_vl['tfidf_score'].values
yt               = df_vl['ground_truth'].values

kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
tiers_for_cv = df_vl['original_label'].values
weight_results = {}

for w in np.round(np.arange(0.50, 1.01, 0.05), 2):
    fold_scores = []
    for _, val_idx in kf.split(np.zeros(len(df_vl)), tiers_for_cv):
        ens = w * best_sbert_preds[val_idx] + (1-w) * tfidf_arr[val_idx]
        sp  = spearmanr(ens, yt[val_idx]).correlation
        fold_scores.append(sp)
    weight_results[w] = round(float(np.mean(fold_scores)), 4)

BEST_W = max(weight_results, key=weight_results.get)
print(f'\nCV Weight Grid:')
for w, s in sorted(weight_results.items()):
    marker = ' <-- BEST' if w == BEST_W else ''
    print(f'  SBERT weight={w:.2f}  Avg Spearman={s:.4f}{marker}')
print(f'\nOptimal SBERT weight: {BEST_W:.2f} | TF-IDF weight: {1-BEST_W:.2f}')

# ── Compute final ensemble scores ────────────────────────────────────
def ensemble_score(s, t, sw=BEST_W):
    return round((sw*s + (1-sw)*t)*100, 2)

def get_tier(score):
    if score >= 70:   return 'Strong Match'
    elif score >= 45: return 'Potential Fit'
    else:             return 'Poor Match'

df_vl['ensemble_score'] = [
    round(BEST_W*s + (1-BEST_W)*t, 2)
    for s,t in zip(best_sbert_preds, tfidf_arr)
]
df_vl['tier'] = df_vl['ensemble_score'].apply(get_tier)
df_sample = df_vl.copy()
print('\nEnsemble scoring complete.')
df_sample[['mpnet_score','roberta_score','jina_score',
           'tfidf_score','ensemble_score','tier']].describe()

In [ ]:
def safe_r(x, y, kind='pearson'):
    if len(x)<2 or np.std(x)==0 or np.std(y)==0: return 0.0
    v = pearsonr(x,y)[0] if kind=='pearson' else spearmanr(x,y).correlation
    return 0.0 if np.isnan(v) else float(v)

def precision_at_k(yt, yp, k=5, thresh=0.5):
    top = sorted(zip(yp,yt), reverse=True)[:k]
    return sum(1 for _,t in top if t>=thresh)/k

def mrr_score(yt, yp, thresh=0.5):
    for rank,(_,t) in enumerate(sorted(zip(yp,yt),reverse=True),1):
        if t>=thresh: return 1.0/rank
    return 0.0

def full_metrics(yt_norm, yp_pct, name, yt_raw=None):
    yt  = np.asarray(yt_norm, float)
    yp  = np.asarray(yp_pct,  float)
    ypr = np.clip(yp/100, 0, 1)
    ytr = denormalize_score(yt) if yt_raw is None else np.asarray(yt_raw, float)
    ydp = denormalize_score(ypr)
    mask = ~np.isnan(yt) & ~np.isnan(yp)
    yt, yp, ypr, ytr, ydp = yt[mask], yp[mask], ypr[mask], ytr[mask], ydp[mask]
    if len(yt)<2: return {}
    sp   = safe_r(ypr,yt,'spearman')
    pe   = safe_r(ypr,yt,'pearson')
    mae  = mean_absolute_error(yt,ypr)
    maer = mean_absolute_error(ytr,ydp)
    rmse = float(np.sqrt(np.mean((ytr-ydp)**2)))
    r2   = float(r2_score(yt,ypr))
    nd   = ndcg_score([yt],[yp])
    p5   = precision_at_k(yt,ypr,5)
    p10  = precision_at_k(yt,ypr,10)
    m    = mrr_score(yt,ypr)
    print(f'{name:18s} Spearman={sp:+.4f} Pearson={pe:+.4f} '
          f'MAE={mae:.4f} MAE_raw={maer:.2f} RMSE={rmse:.2f} '
          f'R2={r2:+.4f} NDCG={nd:.4f} P@5={p5:.2f} MRR={m:.4f}')
    return dict(model=name, spearman=sp, pearson=pe, mae=mae,
                mae_raw=maer, rmse_raw=rmse, r2_raw=r2, ndcg=nd,
                precision_at_5=p5, precision_at_10=p10, mrr=m)

print('Full metrics on validation set:')
yt    = df_sample['ground_truth'].values
yt_r  = df_sample['ats_score_raw'].values
r_mpnet    = full_metrics(yt, df_sample['mpnet_score'].values,    'mpnet', yt_r)
r_roberta  = full_metrics(yt, df_sample['roberta_score'].values,  'roberta', yt_r)
r_jina     = full_metrics(yt, df_sample['jina_score'].values,     'jina', yt_r)
r_tfidf    = full_metrics(yt, df_sample['tfidf_score'].values,    'TF-IDF', yt_r)
r_ensemble = full_metrics(yt, df_sample['ensemble_score'].values, f'Ensemble({best_sbert_name}+TF-IDF)', yt_r)

metrics_df = pd.DataFrame([r_mpnet,r_roberta,r_jina,r_tfidf,r_ensemble])
if 'model' in metrics_df.columns:
    metrics_df = metrics_df.set_index('model')
print('\nSummary Table:')
display(metrics_df.round(4))

In [ ]:
# Scoring helpers for the Gradio dashboard

def extract_pdf_text(path):
    try: return ' '.join(p.get_text() for p in fitz.open(path)).strip()
    except Exception as e: return f'PDF error: {e}'

def extract_keywords(text, n=30):
    try:
        vec = TfidfVectorizer(stop_words='english', max_features=n)
        vec.fit([text])
        return set(vec.get_feature_names_out())
    except Exception:
        tokens = word_tokenize(text.lower())
        return set(w for w,_ in Counter(
            t for t in tokens if t not in STOP_WORDS and len(t)>2
        ).most_common(n))

def sbert_score_best(resume, jd):
    """Score using the winning SBERT model with chunking (if 512-token) or full text (Jina)."""
    return score_doc_pair(best_sbert_model, resume, jd)

def score_single(resume, jd, sw=BEST_W):
    s = sbert_score_best(resume, jd)
    t = tfidf_score(clean_text(resume), clean_text(jd))
    return round(s*100,2), round(t*100,2), ensemble_score(s,t,sw)

def tab1_score(resume_file, paste, jd, sw):
    resume = extract_pdf_text(resume_file.name) if resume_file else paste.strip()
    if not resume:     return 'Please upload PDF or paste resume.', None, ''
    if not jd.strip(): return 'Please enter a job description.', None, ''
    s, t, e = score_single(resume, jd, sw)
    tier = get_tier(e)
    fig = go.Figure()
    for val,nm,col in zip([s,t,e],
                           [f'SBERT ({best_sbert_name})','TF-IDF','Ensemble'],
                           ['#4A90D9','#E67E22','#27AE60']):
        fig.add_trace(go.Bar(x=[nm],y=[val],marker_color=col,
                             text=[f'{val:.1f}%'],textposition='outside',name=nm))
    fig.update_layout(title='Score Breakdown',yaxis=dict(range=[0,115]),
                      height=350,showlegend=False,
                      plot_bgcolor='rgba(0,0,0,0)',paper_bgcolor='rgba(0,0,0,0)')
    kr = extract_keywords(resume,40)
    kj = extract_keywords(jd,40)
    matched,missing,extra = kr&kj, kj-kr, kr-kj
    html = (
        "<div style='font-family:sans-serif;padding:12px'>"
        "<h3 style='color:#27AE60'>Matched (" + str(len(matched)) + ")</h3>"
        "<p style='color:#27AE60'>" + (', '.join(sorted(matched)) or 'None') + "</p>"
        "<hr><h3 style='color:#E74C3C'>Missing from Resume (" + str(len(missing)) + ")</h3>"
        "<p style='color:#E74C3C'>" + (', '.join(sorted(missing)) or 'None') + "</p>"
        "<hr><h3 style='color:#F39C12'>Resume-Only (" + str(len(extra)) + ")</h3>"
        "<p style='color:#F39C12'>" + (', '.join(sorted(extra)) or 'None') + "</p></div>"
    )
    summary = (
        f'## {tier}\n\n'
        f'| Model | Score |\n|---|---|\n'
        f'| SBERT ({best_sbert_name}) | {s:.1f}% |\n'
        f'| TF-IDF | {t:.1f}% |\n'
        f'| **Ensemble** | **{e:.1f}%** |\n'
        f'| Keyword Overlap | {len(matched)}/{len(kj)} JD keywords |'
    )
    return summary, fig, html

def tab2_rank(files, jd, sw):
    if not files or not jd.strip(): return None, None
    rows = []
    for rf in files:
        text = extract_pdf_text(rf.name)
        name = os.path.basename(rf.name).replace('.pdf','')
        s, t, e = score_single(text, jd, sw)
        rows.append(dict(Candidate=name,SBERT=s,TF_IDF=t,Ensemble=e,Tier=get_tier(e)))
    rdf = pd.DataFrame(rows).sort_values('Ensemble',ascending=False)
    rdf.insert(0,'Rank',range(1,len(rdf)+1))
    colors = ['#27AE60' if r>=70 else '#F39C12' if r>=45 else '#E74C3C'
              for r in rdf['Ensemble']]
    fig = go.Figure(go.Bar(x=rdf['Candidate'],y=rdf['Ensemble'],
                           marker_color=colors,
                           text=[f'{v:.1f}%' for v in rdf['Ensemble']],
                           textposition='outside'))
    fig.update_layout(title='Candidate Ranking',yaxis=dict(range=[0,115]),
                      height=400,plot_bgcolor='rgba(0,0,0,0)',
                      paper_bgcolor='rgba(0,0,0,0)')
    return rdf[['Rank','Candidate','SBERT','TF_IDF','Ensemble','Tier']], fig

def tab3_heatmap(n=20):
    n   = min(int(n), len(df_sample))
    sub = df_sample.head(n).copy()
    sub['Label'] = [f'C{i+1}' for i in range(n)]
    cols = ['mpnet_score','roberta_score','jina_score','tfidf_score','ensemble_score']
    z    = sub[cols].values.T
    fh   = go.Figure(go.Heatmap(z=z, x=sub['Label'].tolist(),
                                y=['mpnet','roberta','jina','TF-IDF','Ensemble'],
                                colorscale='RdYlGn',zmin=0,zmax=100,
                                text=np.round(z,1),texttemplate='%{text}',
                                colorbar=dict(title='%')))
    fh.update_layout(title=f'Score Heatmap - {n} Candidates',height=380,
                     plot_bgcolor='rgba(0,0,0,0)',paper_bgcolor='rgba(0,0,0,0)')
    fd = go.Figure()
    for col,nm,c in zip(cols,
                         ['mpnet','roberta','jina','TF-IDF','Ensemble'],
                         ['#4A90D9','#9B59B6','#E67E22','#95A5A6','#27AE60']):
        fd.add_trace(go.Histogram(x=df_sample[col],name=nm,opacity=0.6,
                                   marker_color=c,nbinsx=20))
    fd.update_layout(barmode='overlay',title='Score Distribution',height=340,
                     plot_bgcolor='rgba(0,0,0,0)',paper_bgcolor='rgba(0,0,0,0)')
    return fh, fd

def tab4_compare():
    if metrics_df.empty:
        blank = go.Figure().update_layout(title='No metrics.')
        return blank, blank, blank
    models  = metrics_df.index.tolist()
    palette = ['#4A90D9','#9B59B6','#E67E22','#95A5A6','#27AE60']
    mcols   = ['spearman','pearson','ndcg','precision_at_5','precision_at_10','mrr']
    fb = go.Figure()
    for i,m in enumerate(models):
        vals = [metrics_df.loc[m,c] for c in mcols]
        fb.add_trace(go.Bar(name=m,
            x=[c.replace('_',' ').title() for c in mcols],
            y=vals,marker_color=palette[i%len(palette)],
            text=[f'{v:.3f}' for v in vals],textposition='outside'))
    fb.update_layout(barmode='group',title='All Metrics Comparison',
                     yaxis=dict(range=[-0.2,1.3]),height=430,
                     plot_bgcolor='rgba(0,0,0,0)',paper_bgcolor='rgba(0,0,0,0)')
    rcols = ['spearman','ndcg','precision_at_5','precision_at_10','mrr','pearson']
    fr = go.Figure()
    for i,m in enumerate(models):
        vals = [metrics_df.loc[m,c] for c in rcols]+[metrics_df.loc[m,rcols[0]]]
        cats = [c.replace('_',' ').title() for c in rcols+[rcols[0]]]
        fr.add_trace(go.Scatterpolar(r=vals,theta=cats,fill='toself',
                                      name=m,line_color=palette[i%len(palette)],opacity=0.5))
    fr.update_layout(polar=dict(radialaxis=dict(range=[0,1])),
                     title='Radar Chart',height=430,paper_bgcolor='rgba(0,0,0,0)')
    fm = go.Figure(go.Bar(x=models,
        y=[metrics_df.loc[m,'mae'] for m in models],
        marker_color=palette[:len(models)],
        text=[f"{metrics_df.loc[m,'mae']:.4f}" for m in models],
        textposition='outside'))
    fm.update_layout(title='MAE norm (lower=better)',height=350,
                     plot_bgcolor='rgba(0,0,0,0)',paper_bgcolor='rgba(0,0,0,0)')
    return fb, fr, fm

print('Dashboard functions ready')

In [ ]:
fig_bar_cmp, fig_radar_cmp, fig_mae_cmp = tab4_compare()
fig_heatmap, fig_dist = tab3_heatmap(20)

with gr.Blocks(theme=gr.themes.Soft(), title='EZhire') as demo:

    gr.Markdown('# EZhire - Resume-Job Semantic Similarity Scoring')
    gr.Markdown(
        f'Three-model pipeline: mpnet | roberta | jina  '
        f'| Ensemble uses **{best_sbert_name}** + TF-IDF '
        f'| Optimal SBERT weight: **{BEST_W:.2f}**'
    )

    sbert_w = gr.Slider(0.0,1.0,value=float(BEST_W),step=0.05,
                        label='SBERT Weight (CV-optimised default shown)')

    with gr.Tab('Score a Resume'):
        with gr.Row():
            with gr.Column():
                r_pdf   = gr.File(label='Upload PDF Resume',file_types=['.pdf'])
                r_paste = gr.Textbox(label='Or paste resume text',lines=7)
                jd_box  = gr.Textbox(label='Job Description',lines=7)
                btn1    = gr.Button('Analyse Match',variant='primary')
            with gr.Column():
                out_md  = gr.Markdown()
                out_fig = gr.Plot(label='Score Breakdown')
        out_kw = gr.HTML(label='Keyword Overlap')
        btn1.click(tab1_score,[r_pdf,r_paste,jd_box,sbert_w],
                   [out_md,out_fig,out_kw])

    with gr.Tab('Rank Candidates'):
        with gr.Row():
            with gr.Column():
                r_multi  = gr.File(label='Upload Multiple PDFs',
                                   file_count='multiple',file_types=['.pdf'])
                jd_rank  = gr.Textbox(label='Job Description',lines=7)
                btn2     = gr.Button('Rank Candidates',variant='primary')
            with gr.Column():
                rank_fig = gr.Plot()
        rank_tbl = gr.Dataframe(label='Ranked Candidates',interactive=False)
        btn2.click(tab2_rank,[r_multi,jd_rank,sbert_w],[rank_tbl,rank_fig])

    with gr.Tab('Score Heatmap'):
        n_sl   = gr.Slider(5,min(50,len(df_sample)),value=20,step=5,
                           label='Candidates to show')
        btn3   = gr.Button('Refresh')
        h_plot = gr.Plot(value=fig_heatmap)
        d_plot = gr.Plot(value=fig_dist)
        btn3.click(tab3_heatmap,[n_sl],[h_plot,d_plot])

    with gr.Tab('Model Comparison'):
        gr.Markdown(
            'All three models evaluated on the full validation set. '
            'Ensemble uses the best SBERT (auto-selected by Pearson) '
            'with CV-optimised TF-IDF weight.'
        )
        if not metrics_df.empty:
            gr.Dataframe(value=metrics_df.round(4).reset_index(),label='Metrics Table')
        gr.Plot(value=fig_bar_cmp,  label='All Metrics')
        gr.Plot(value=fig_radar_cmp,label='Radar Chart')
        gr.Plot(value=fig_mae_cmp,  label='MAE')
        gr.Markdown(
            '**Spearman/Pearson**: correlation with ATS ground truth. '
            '**NDCG**: ranking quality. **Precision@K**: good fits in top-K. '
            '**MAE**: prediction error. **RMSE/R2**: ATS-scale accuracy.'
        )

demo.launch(share=True, debug=True)